# Detecting protest events on my data

In [1]:
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
device

'cuda:0'

In [2]:
from protest_impact.data.protests.detection import load_glpn_dataset

glpn = load_glpn_dataset()
glpn = glpn.rename_column("excerpt", "text")

Using custom data configuration default-8ba850f9ef63b7f4
Found cached dataset csv (/root/.cache/huggingface/datasets/csv/default-8ba850f9ef63b7f4/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317)


  0%|          | 0/5 [00:00<?, ?it/s]

Loading cached processed dataset at /root/.cache/huggingface/datasets/csv/default-8ba850f9ef63b7f4/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-23773ba0ca7bb6ee.arrow
Loading cached processed dataset at /root/.cache/huggingface/datasets/csv/default-8ba850f9ef63b7f4/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-5a400a3dcacfe1d4.arrow
Loading cached processed dataset at /root/.cache/huggingface/datasets/csv/default-8ba850f9ef63b7f4/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-2fc85313f4b3bbf9.arrow
Loading cached processed dataset at /root/.cache/huggingface/datasets/csv/default-8ba850f9ef63b7f4/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-a712e1851e42fa57.arrow
Loading cached processed dataset at /root/.cache/huggingface/datasets/csv/default-8ba850f9ef63b7f4/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-5be37dd13ef1b218.arrow


In [3]:
from protest_impact.data.protests.detection import load_aglpn_dataset

protest_news = load_aglpn_dataset()
protest_news, protest_news["train"].features

Using custom data configuration protest-5e5518d3acc2913b
Found cached dataset json (/root/.cache/huggingface/datasets/json/protest-5e5518d3acc2913b/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51)


  0%|          | 0/1 [00:00<?, ?it/s]

Loading cached processed dataset at /root/.cache/huggingface/datasets/json/protest-5e5518d3acc2913b/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-7e15eee90aca284a.arrow
Loading cached split indices for dataset at /root/.cache/huggingface/datasets/json/protest-5e5518d3acc2913b/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-d4311965ef7bebe2.arrow and /root/.cache/huggingface/datasets/json/protest-5e5518d3acc2913b/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-a3ca6af25a187a06.arrow
Loading cached split indices for dataset at /root/.cache/huggingface/datasets/json/protest-5e5518d3acc2913b/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-e1c767c42d3cf224.arrow and /root/.cache/huggingface/datasets/json/protest-5e5518d3acc2913b/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-0574c733845f527e.arrow
Loading cached processed dataset at /root/.cache

(DatasetDict({
     train: Dataset({
         features: ['text', 'meta', '_input_hash', '_task_hash', 'spans', 'options', 'accept', '_view_id', 'config', 'answer', '_timestamp', 'label'],
         num_rows: 575
     })
     dev: Dataset({
         features: ['text', 'meta', '_input_hash', '_task_hash', 'spans', 'options', 'accept', '_view_id', 'config', 'answer', '_timestamp', 'label'],
         num_rows: 115
     })
     test: Dataset({
         features: ['text', 'meta', '_input_hash', '_task_hash', 'spans', 'options', 'accept', '_view_id', 'config', 'answer', '_timestamp', 'label'],
         num_rows: 460
     })
 }),
 {'text': Value(dtype='string', id=None),
  'meta': {'date': Value(dtype='string', id=None),
   'title': Value(dtype='string', id=None),
   'url': Value(dtype='string', id=None),
   'homepage': Value(dtype='string', id=None),
   'crawl_engine': Value(dtype='string', id=None),
   'crawl_query': Value(dtype='string', id=None),
   'pattern': Value(dtype='string', id=None)

In [4]:
len(glpn["train"]), len(protest_news["train"])

(1914, 575)

In [5]:
import spacy

from protest_impact.data.protests.config import search_regex

nlp = spacy.load("de_core_news_sm", disable=["parser", "tagger", "ner", "tokenizer"])
nlp.add_pipe("sentencizer")


def kwic(text, n=0):
    sents = list(nlp(text).sents)
    kwics_nrs = set()
    for i, sent in enumerate(sents):
        if search_regex.search(sent.text_with_ws):
            kwics_nrs.add(i)
            for j in range(1, n + 1):
                kwics_nrs.add(i - j)
                kwics_nrs.add(i + j)
    kwic_text = ""
    for kwic_nr in sorted(list(kwics_nrs)):
        if kwic_nr >= 0 and kwic_nr < len(sents):
            if kwic_nr - 1 not in kwics_nrs:
                kwic_text += "\n...\n"
            kwic_text += sents[kwic_nr].text_with_ws
    return kwic_text

In [6]:
def kwic_dataset(dataset, n=0):
    return dataset.map(
        lambda x: {
            "text": x["meta"]["title"]
            + "\n\n"
            + kwic("\n".join(list(x["text"].split("\n"))[1:]), n=n)
        }
    )

In [7]:
protest_news = kwic_dataset(protest_news, n=2)

  0%|          | 0/575 [00:00<?, ?ex/s]

  0%|          | 0/115 [00:00<?, ?ex/s]

  0%|          | 0/460 [00:00<?, ?ex/s]

In [8]:
if device == "cpu":
    n = 20
    glpn["train"] = glpn["train"].shuffle(seed=20230206).select(range(n))
    glpn["dev"] = glpn["dev"].shuffle(seed=20230206).select(range(n))
    glpn["test"] = glpn["test"].shuffle(seed=20230206).select(range(n))
    glpn["test.time"] = glpn["test.time"].shuffle(seed=20230206).select(range(n))
    glpn["test.loc"] = glpn["test.loc"].shuffle(seed=20230206).select(range(n))

    protest_news["train"] = (
        protest_news["train"].shuffle(seed=20230206).select(range(n))
    )
    protest_news["dev"] = protest_news["dev"].shuffle(seed=20230206).select(range(n))
    protest_news["test"] = protest_news["test"].shuffle(seed=20230206).select(range(n))

In [9]:
import evaluate
import numpy as np

metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [10]:
from pathlib import Path

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)


def train_model(model, tokenizer, description, dataset):
    model_location = f"models/{description}"

    def tokenize_function(examples):
        return tokenizer(
            examples["text"], padding="max_length", truncation=True, max_length=512
        )

    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    training_args = TrainingArguments(
        output_dir=f"{model_location}/checkpoints",
        evaluation_strategy="epoch",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        learning_rate=5e-6,
        weight_decay=0.2,
        num_train_epochs=6,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["dev"],
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model(f"{model_location}/model")
    return model

In [11]:
model_name = "deepset/gelectra-base"
# model_name = "deepset/gelectra-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# model_vanilla = AutoModelForSequenceClassification.from_pretrained(model_name).to(
    # device
# )

In [12]:
!pwd

/workspace/persistent/protest-impact/docs/notes


In [13]:
!ls models/deepset-gelectra-large-protest-news/model/

config.json  pytorch_model.bin	training_args.bin


In [14]:
#model_glpn = train_model(
#    model_vanilla, tokenizer, f"{model_name.replace('/', '-')}-glpn", glpn
#)
model_glpn = AutoModelForSequenceClassification.from_pretrained(
    "models/deepset-gelectra-large-glpn").to(device)

In [15]:
from evaluate import TextClassificationEvaluator


def evaluate(model, tokenizer, test_set):
    eval_results = TextClassificationEvaluator().compute(
        model_or_pipeline=model,
        data=test_set,
        input_column="text",
        label_column="label",
        label_mapping={"LABEL_0": 0, "LABEL_1": 1},
        # label_mapping={"irrelevant": 0, "relevant": 1},
        # label_mapping={"LABEL_0": "irrelevant", "LABEL_1": "relevant"},
        tokenizer=tokenizer,
        metric=metric,
        average=None
    )
    return eval_results

In [21]:
from sklearn.metrics import classification_report
from transformers import TextClassificationPipeline

pipe = TextClassificationPipeline(model=model_glpn, tokenizer=tokenizer, 
    padding="max_length", truncation=True, max_length=512,
    device=device)
preds = pipe(protest_news["test"]["text"])
labels = [int(p['label'][-1]) for p in preds]
print(classification_report(labels,  protest_news["test"]['label']))

              precision    recall  f1-score   support

           0       0.81      0.72      0.76       318
           1       0.50      0.62      0.55       142

    accuracy                           0.69       460
   macro avg       0.65      0.67      0.66       460
weighted avg       0.71      0.69      0.70       460



In [22]:
# evaluate(model_glpn, tokenizer, protest_news["test"])

TypeError: TextClassificationEvaluator.compute() got an unexpected keyword argument 'average'

In [23]:
# model_protest_news = train_model(
#     model_vanilla,
#     tokenizer,
#     f"{model_name.replace('/', '-')}-protest-news",
#     protest_news,
# )

In [ ]:
# evaluate(model_protest_news, tokenizer, protest_news["test"])

In [24]:
model_glpn_protest_news = train_model(
    model_glpn,
    tokenizer,
    f"{model_name.replace('/', '-')}-glpn-protest-news",
    protest_news,
)

  0%|          | 0/1 [00:00<?, ?ba/s]

  0%|          | 0/1 [00:00<?, ?ba/s]

  0%|          | 0/1 [00:00<?, ?ba/s]

The following columns in the training set don't have a corresponding argument in `ElectraForSequenceClassification.forward` and have been ignored: options, text, accept, _task_hash, _view_id, meta, config, _timestamp, spans, _input_hash, answer. If options, text, accept, _task_hash, _view_id, meta, config, _timestamp, spans, _input_hash, answer are not expected by `ElectraForSequenceClassification.forward`,  you can safely ignore this message.
/workspace/persistent/protest-impact/.venv/lib/python3.10/site-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 575
  Num Epochs = 6
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 4
  Gradient Accumulation steps =

Epoch,Training Loss,Validation Loss,F1
1,No log,0.693273,0.592593
2,No log,0.631363,0.545455
3,No log,0.805783,0.545455
4,0.753900,1.049122,0.439024
5,0.753900,1.366469,0.431818
6,0.753900,1.490261,0.436782


The following columns in the evaluation set don't have a corresponding argument in `ElectraForSequenceClassification.forward` and have been ignored: options, text, accept, _task_hash, _view_id, meta, config, _timestamp, spans, _input_hash, answer. If options, text, accept, _task_hash, _view_id, meta, config, _timestamp, spans, _input_hash, answer are not expected by `ElectraForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 115
  Batch size = 4
The following columns in the evaluation set don't have a corresponding argument in `ElectraForSequenceClassification.forward` and have been ignored: options, text, accept, _task_hash, _view_id, meta, config, _timestamp, spans, _input_hash, answer. If options, text, accept, _task_hash, _view_id, meta, config, _timestamp, spans, _input_hash, answer are not expected by `ElectraForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****


In [ ]:
# evaluate(model_glpn_protest_news, tokenizer, protest_news["test"])

In [25]:
pipe = TextClassificationPipeline(model=model_glpn_protest_news, tokenizer=tokenizer, 
    padding="max_length", truncation=True, max_length=512,
    device=device)
preds = pipe(protest_news["test"]["text"])
labels = [int(p['label'][-1]) for p in preds]
print(classification_report(labels,  protest_news["test"]['label']))

              precision    recall  f1-score   support

           0       0.77      0.69      0.73       315
           1       0.45      0.54      0.49       145

    accuracy                           0.65       460
   macro avg       0.61      0.62      0.61       460
weighted avg       0.67      0.65      0.65       460



In [26]:
pipe = TextClassificationPipeline(model=model_glpn, tokenizer=tokenizer, 
    padding="max_length", truncation=True, max_length=512,
    device=device)

In [31]:
import json

with open( "../../protest_news_shuffled_v2_sample.jsonl") as f:
    news = [json.loads(line) for line in f]


In [32]:
from tqdm import tqdm

for i in tqdm(range(200)):
    data = news[100*i:100*(i+1)]
    data = [kwic(a['text'], n=2) for a in data]
    preds = pipe(data)
    with open(f"somefile-{i}.jsonl", 'w') as f:
        json.dump(preds, f)

  5%|▌         | 10/200 [01:07<22:18,  7.05s/it]/workspace/persistent/protest-impact/.venv/lib/python3.10/site-packages/transformers/pipelines/base.py:1045: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
100%|██████████| 200/200 [21:43<00:00,  6.52s/it]
